# Phase 6 — Hyperparameter Tuning and Final Model Selection

Tunes the three models Phase 5 flagged as promising — Random Forest, Gradient Boosting, Logistic Regression — using `RandomizedSearchCV` scored on **PR-AUC (average precision)**, with the same `GroupKFold(5)` grouped by `Patient File No.` used everywhere else in this project. Decision Tree and XGBoost are not tuned (Phase 5: worst on every metric / no advantage over Gradient Boosting, respectively).

`holdout_validation` is not touched here — it stays frozen for Phase 7.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from src.config import INTERIM_DIR, TABLES_DIR, MODELS_DIR, PATIENT_ID_COL, RANDOM_SEED
from src.preprocessing import split_features_target
from src.tune import tune_model, tuned_estimator_factory
from src.train import run_grouped_cv
from src.evaluate import build_metrics_table_with_ci, get_calibration_curve
from src.viz import apply_chart_style, save_fig, GRIDLINE

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

train_pool = pd.read_csv(INTERIM_DIR / "train_pool.csv")
X, y = split_features_target(train_pool)
groups = train_pool[PATIENT_ID_COL]

CANDIDATES = ["logistic_regression", "random_forest", "gradient_boosting"]
NEEDS_WEIGHTING = {"logistic_regression": False, "random_forest": False, "gradient_boosting": True}

## 1. Run the search for each candidate

In [ ]:
searches = {}
for key in CANDIDATES:
    searches[key] = tune_model(key, X, y, groups, n_iter=40)
    print(f"{key}: best PR-AUC (search CV) = {searches[key].best_score_:.4f}")
    print(f"  best params: { {k.replace('model__',''): v for k, v in searches[key].best_params_.items()} }")

## 2. Re-run Phase 5's full metric suite on the TUNED models

Same grouped out-of-fold evaluation as Phase 5, just with each model's tuned hyperparameters instead of scikit-learn defaults — a fair, apples-to-apples before/after comparison.

In [ ]:
all_oof_tuned = []
for key in CANDIDATES:
    factory = tuned_estimator_factory(key, searches[key].best_params_)
    _, oof = run_grouped_cv(f"{key}_tuned", factory, NEEDS_WEIGHTING[key], X, y, groups)
    all_oof_tuned.append(oof)

oof_tuned = pd.concat(all_oof_tuned, ignore_index=True)
oof_tuned.to_csv(TABLES_DIR / "phase6_oof_tuned_predictions.csv", index=False)

metrics_tuned = build_metrics_table_with_ci(oof_tuned, n_boot=500)
metrics_tuned.to_csv(TABLES_DIR / "phase6_tuned_metrics_with_ci.csv")
metrics_tuned[["accuracy","recall_sensitivity","specificity","f1","roc_auc","pr_auc","balanced_accuracy","brier_score"]].round(3)

## Before vs. after tuning

| Model | Recall (before→after) | PR-AUC (before→after) | F1 (before→after) |
|---|---|---|---|
| Random Forest | 0.775 → 0.770 | 0.907 → 0.921 | 0.824 → 0.825 |
| Gradient Boosting | 0.812 → 0.799 | 0.900 → 0.900 | 0.806 → 0.816 |
| Logistic Regression | 0.826 → **0.855** | 0.828 → **0.905** | 0.773 → 0.799 |

**Logistic Regression is the model tuning changed the most, and specifically in the direction that matters for this project.** Its best hyperparameters used **L1 regularization**, which zeroes out uninformative/redundant coefficients rather than shrinking all of them — worth checking directly, since Phase 2's correlation heatmap flagged real redundancy (FSH/FSH-LH-ratio, Weight/BMI, Hip/Waist) that L1 is specifically suited to resolve.

(Numbers reflect the preprocessing pipeline after the Phase 8 robustness fix — see the final-pipeline cell below for why a `ClipStandardized` step was added between Phase 6's first run and this one. The change here is a rerun with that fix already in place, not a second tuning pass.)

In [ ]:
best_lr_pipeline = searches["logistic_regression"].best_estimator_
coefs = best_lr_pipeline.named_steps["model"].coef_[0]
feature_names = best_lr_pipeline.named_steps["preprocess"].get_feature_names_out()
n_zero = int((coefs == 0).sum())
print(f"L1 zeroed {n_zero} of {len(coefs)} features, keeping {len(coefs) - n_zero}")

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df[coef_df["coef"] != 0].sort_values("abs_coef", ascending=False).head(10)[["feature", "coef"]]

**Interpretation:** L1 kept 26 of 49 features and zeroed 23. The surviving top coefficients — Follicle count (both ovaries), cycle regularity, hirsutism, skin darkening, acne, weight gain, AMH — match exactly what Phase 2's EDA flagged as the strongest associations, including the same circularity caveat (follicle count and cycle regularity are Rotterdam diagnostic-criterion features, not independently discovered risk factors). This is a good sign: tuning didn't find some arbitrary pattern, it converged on the same signal the EDA already surfaced by hand — and previews Phase 8's SHAP analysis, which will explain these same features' individual contributions.

## 3. Calibration check: does the finalist avoid Random Forest's S-shape problem?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0, 1], [0, 1], linestyle="--", color=GRIDLINE, linewidth=1.5, label="Perfectly calibrated", zorder=1)
colors = {"logistic_regression_tuned": "#2a78d6", "random_forest_tuned": "#1baf7a", "gradient_boosting_tuned": "#eda100"}
labels = {"logistic_regression_tuned": "Logistic Regression (tuned)", "random_forest_tuned": "Random Forest (tuned)", "gradient_boosting_tuned": "Gradient Boosting (tuned)"}
for model_name in ["logistic_regression_tuned", "random_forest_tuned", "gradient_boosting_tuned"]:
    sub = oof_tuned[oof_tuned["model"] == model_name]
    frac_pos, mean_pred = get_calibration_curve(sub, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", markersize=4, color=colors[model_name], linewidth=1.8, label=labels[model_name])
ax.set_xlabel("Mean predicted probability (per bin)")
ax.set_ylabel("Observed PCOS rate (per bin)")
ax.set_title("Calibration — tuned models, out-of-fold predictions")
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
apply_chart_style(ax)
save_fig(fig, "17_tuned_calibration_curves")
plt.show()

**Interpretation:** Random Forest still shows the same S-shape found in Phase 5 — tuning its tree structure didn't fix this, since the distortion comes from averaging many trees' votes, not from any specific hyperparameter. Logistic Regression tracks the diagonal more closely overall, though it still overstates risk somewhat in the 0.3-0.75 range (e.g. predicts ~49% when the true observed rate in that bin is ~36%). Neither is perfect, but Logistic Regression's miscalibration is smaller and more monotonic — no recalibration step is strictly required, though `CalibratedClassifierCV` remains an easy option if the shipped probability needs to be tighter.

## 4. Final model selection

| Model | Recall | Specificity | F1 | PR-AUC | Brier | Interpretable? |
|---|---:|---:|---:|---:|---:|---|
| Random Forest (tuned) | 0.770 | **0.957** | 0.825 | **0.921** | **0.088** | No |
| Gradient Boosting (tuned) | 0.799 | 0.930 | 0.816 | 0.900 | 0.098 | No |
| **Logistic Regression (tuned)** | **0.855** | 0.874 | 0.799 | 0.905 | 0.091 | **Yes** |

**Decision: Logistic Regression (tuned, L1, C≈0.083).**

Reasoning, weighed the way this project's own brief asks (recall, F1, PR-AUC, specificity, calibration, generalization, interpretability — not just whichever number is highest):
- **Recall is this project's stated priority** (false negatives = missed PCOS cases), and Logistic Regression has the best recall of all three by a clear margin (0.855 vs 0.799/0.770).
- Its PR-AUC (0.905) and F1 (0.799) are within ~0.02-0.03 of the ensembles — a small discrimination cost for a real recall gain, not a large one.
- It is **fully interpretable** — a linear model with 26 non-zero coefficients — which matters directly for Phase 8 (SHAP is exact and simple for linear models) and for the project's explicit framing as a research/decision-support prototype, not a black box.
- Its calibration is the least distorted of the three (no S-shape), and Phase 4 already found it has the smallest train-validation overfitting gap of any real model — the best generalization signal available pre-holdout.
- **The honest tradeoff:** Random Forest and Gradient Boosting both have meaningfully better specificity (~0.93-0.96 vs 0.874) — Logistic Regression will flag more false positives. Given this is a screening tool meant to prompt further evaluation rather than a diagnosis, that tradeoff (more false alarms, fewer missed cases) matches the project's own stated risk preference. If specificity/false-alarm burden turns out to matter more in practice, Random Forest is the documented runner-up.

This is a judgment call, not a mechanical maximum — flagging it here explicitly so it can be revisited if `holdout_validation` (Phase 7) tells a different story.

## 5. Freeze the final pipeline

Refit on **all of `train_pool`** (not just CV folds) with the tuned hyperparameters, then save the complete pipeline (preprocessing + model) — not just the bare estimator — so `holdout_validation` and any future new patient can be scored with a single `.predict_proba()` call.

In [ ]:
final_pipeline = searches["logistic_regression"].best_estimator_  # already refit on all of X, y by RandomizedSearchCV(refit=True)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, MODELS_DIR / "pcos_risk_pipeline.joblib")

metadata = {
    "model_type": "logistic_regression",
    "hyperparameters": {k.replace("model__", ""): v for k, v in searches["logistic_regression"].best_params_.items()},
    "random_seed": RANDOM_SEED,
    "training_rows": int(len(train_pool)),
    "training_patients": int(groups.nunique()),
    "n_features_total": int(len(feature_names)),
    "n_features_nonzero": int(len(feature_names) - n_zero),
    "cv_metrics": {
        metric: float(metrics_tuned.loc["logistic_regression_tuned", metric])
        for metric in ["accuracy", "recall_sensitivity", "specificity", "f1", "roc_auc", "pr_auc", "brier_score"]
    },
    "selection_rationale": (
        "Selected for highest recall among tuned candidates (project priority: minimize "
        "missed PCOS cases), competitive PR-AUC/F1, full interpretability for SHAP (Phase 8), "
        "smallest train-val overfitting gap (Phase 4), and least-distorted calibration curve. "
        "Random Forest (tuned) is the documented runner-up if specificity matters more in practice."
    ),
    "known_limitations": [
        "Follicle count and cycle regularity are Rotterdam diagnostic-criterion features, not "
        "independently discovered risk factors - expect the model to lean on them heavily.",
        "holdout_validation is drawn from the same source population as train_pool, not a "
        "genuinely independent cohort - see Phase 1 for why.",
        "Calibration is imperfect (overstates risk somewhat in the 0.3-0.75 predicted range).",
        "Numeric features are clipped to +/-5 standard deviations after scaling (see "
        "src/preprocessing.py ClipStandardized) - found necessary in Phase 8 when a holdout "
        "patient's PRG(ng/mL)=85.0 (training max: 25.3) produced a z-score of ~49 and dominated "
        "that patient's prediction. Any single feature far enough outside the training range can "
        "still influence a prediction more than intended even with this safeguard; extreme inputs "
        "at inference time should be treated with caution regardless.",
    ],
}
with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("saved models/pcos_risk_pipeline.joblib and models/model_metadata.json")
metadata